In [47]:
import pandas as pd
import geopandas as gpd
import json
import os
import numpy as np
import shapely
import matplotlib.pyplot as plt


In [48]:
#DATA_RAW = 'https://storage.googleapis.com/wetlands-gap-map/original_data/protected_areas_name_unique_final.gpkg'
DATA_FOLDER = '../data/raw/European Wetlands Atlas'
VECTOR_FOLDER = '../data/raw/vectors'
OUTPUT_MBTILES = '../data/processed/mbtiles'


## Protected Areas

In [49]:
pa_raw = gpd.read_file(DATA_FOLDER + '/protected_areas_name_unique_final.gpkg', layer=0)
pa_raw = pa_raw[['NAME_UNIQUE', 'DESIG_ENG', 'tier', 'Natura 2000', 'Ramsar site', 'geometry']]
pa_raw.columns = pa_raw.columns.str.lower().str.replace(' ', '_')
pa_raw = pa_raw.to_crs('EPSG:4326')

pa_raw.head()

,name_unique,desig_eng,tier,natura_2000,ramsar_site,geometry
0,NSG Emsaue,Nature Reserve,B - National and regional protected areas,None,None,"MULTIPOLYGON (((7.61935 52.0822, 7.61965 52.08..."
1,NSG Lippeaue,Nature Reserve,B - National and regional protected areas,None,None,"MULTIPOLYGON (((6.63561 51.63785, 6.63547 51.6..."
2,Mittlere Elbe,Landscape Protection Area,B - National and regional protected areas,None,None,"MULTIPOLYGON (((11.95172 51.934, 11.9514 51.93..."
3,Aspskär Islands,Wetland of International Importance (Ramsar),A - Core international,None,yes,"MULTIPOLYGON (((26.43076 60.26086, 26.41382 60..."
4,Söderskär and Långören Archipelago,Wetland of International Importance (Ramsar),A - Core international,None,yes,"MULTIPOLYGON (((25.90243 60.11267, 25.81167 60..."


In [50]:
pa_clean = pa_raw.copy()
pa_clean['natura_2000'] = pa_clean['natura_2000'].apply(lambda x: 1 if x == 'yes' else 0)
pa_clean['ramsar_site'] = pa_clean['ramsar_site'].apply(lambda x: 1 if x == 'yes' else 0)
#pa_clean['geometry'] = pa_clean['geometry'].simplify(tolerance=0.005, preserve_topology=True)
pa_clean['bbox'] = pa_clean['geometry'].apply(lambda x: x.bounds)
pa_clean['bbox'] = pa_clean['bbox'].apply(lambda x: [x[0], x[1], x[2], x[3]])

pa_clean.head()

,name_unique,desig_eng,tier,natura_2000,ramsar_site,geometry,bbox
0,NSG Emsaue,Nature Reserve,B - National and regional protected areas,0,0,"MULTIPOLYGON (((7.61935 52.0822, 7.61965 52.08...","[7.402533663999974, 51.94423392935471, 8.10935..."
1,NSG Lippeaue,Nature Reserve,B - National and regional protected areas,0,0,"MULTIPOLYGON (((6.63561 51.63785, 6.63547 51.6...","[6.603928666999991, 51.625832627285966, 8.3667..."
2,Mittlere Elbe,Landscape Protection Area,B - National and regional protected areas,0,0,"MULTIPOLYGON (((11.95172 51.934, 11.9514 51.93...","[11.67717983699998, 51.710545950304486, 12.508..."
3,Aspskär Islands,Wetland of International Importance (Ramsar),A - Core international,0,1,"MULTIPOLYGON (((26.43076 60.26086, 26.41382 60...","[26.350482843000066, 60.24320147951152, 26.430..."
4,Söderskär and Långören Archipelago,Wetland of International Importance (Ramsar),A - Core international,0,1,"MULTIPOLYGON (((25.90243 60.11267, 25.81167 60...","[25.35880096700004, 60.018992389492624, 25.902..."


### Create Mapbox layer

In [ ]:
pa_clean.to_file(VECTOR_FOLDER + '/protected_areas_europe.geojson', driver='GeoJSON')

In [8]:
ramsar_clean = pa_clean[pa_clean['ramsar_site'] == 1]
ramsar_clean = ramsar_clean.dropna(subset=['geometry'])
ramsar_clean.to_file(VECTOR_FOLDER + '/ramsar_sites_europe.geojson', driver='GeoJSON')

In [ ]:
!tippecanoe -o {OUTPUT_MBTILES + '/protected_areas_europe_2.mbtiles'} -zg --extend-zooms-if-still-dropping --force -l protected_areas_europe -f {VECTOR_FOLDER + '/protected_areas_europe.geojson'}

In [9]:
!tippecanoe -o {OUTPUT_MBTILES + '/ramsar_sites_europe.mbtiles'} -zg --extend-zooms-if-still-dropping --force -l ramsar_sites_europe -f {VECTOR_FOLDER + '/ramsar_sites_europe.geojson'}

896 features, 27601122 bytes of geometry and attributes, 105905 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z1 for features typically 289774 feet (88323 meters) apart, and at least 59606 feet (18168 meters) apart
Choosing a maxzoom of -z13 for resolution of about 53 feet (16 meters) within features
  99.9%  13/4152/2786  


### Create locations table

In [51]:
locations_pa = pa_clean.copy()
locations_pa.reset_index(drop=False, inplace=True)
locations_pa['id'] = 'europe_wdpa_' + locations_pa['index'].astype(str)
locations_pa.rename(columns={'name_unique': 'name'}, inplace=True)
locations_pa['region'] = 'europe'
locations_pa['code'] = locations_pa['id'] + '_' + locations_pa['name'].str.lower().str.replace(' ', '_')
locations_pa['type'] = 'wdpa'
locations_pa = locations_pa[['id', 'name', 'region', 'code', 'geometry', 'bbox', 'type']]
locations_pa.head()


,id,name,region,code,geometry,bbox,type
0,europe_wdpa_0,NSG Emsaue,europe,europe_wdpa_0_nsg_emsaue,"MULTIPOLYGON (((7.61935 52.0822, 7.61965 52.08...","[7.402533663999974, 51.94423392935471, 8.10935...",wdpa
1,europe_wdpa_1,NSG Lippeaue,europe,europe_wdpa_1_nsg_lippeaue,"MULTIPOLYGON (((6.63561 51.63785, 6.63547 51.6...","[6.603928666999991, 51.625832627285966, 8.3667...",wdpa
2,europe_wdpa_2,Mittlere Elbe,europe,europe_wdpa_2_mittlere_elbe,"MULTIPOLYGON (((11.95172 51.934, 11.9514 51.93...","[11.67717983699998, 51.710545950304486, 12.508...",wdpa
3,europe_wdpa_3,Aspskär Islands,europe,europe_wdpa_3_aspskär_islands,"MULTIPOLYGON (((26.43076 60.26086, 26.41382 60...","[26.350482843000066, 60.24320147951152, 26.430...",wdpa
4,europe_wdpa_4,Söderskär and Långören Archipelago,europe,europe_wdpa_4_söderskär_and_långören_archipelago,"MULTIPOLYGON (((25.90243 60.11267, 25.81167 60...","[25.35880096700004, 60.018992389492624, 25.902...",wdpa


## Admin boundaries

In [52]:
locations_global = gpd.read_file(VECTOR_FOLDER + '/europe_region_complete.geojson')
locations_global['id'] = ['europe_global']
locations_global['name'] = ['Europe']
locations_global['type'] = ['global']
locations_global['region'] = ['europe']
locations_global['code'] = ['europe']
#locations_global['geometry'] = locations_global['geometry'].simplify(tolerance=0.005, preserve_topology=True)
locations_global['bbox'] = locations_global['geometry'].apply(lambda x: x.bounds)
locations_global['bbox'] = locations_global['bbox'].apply(lambda x: [x[0], x[1], x[2], x[3]])

locations_global = locations_global[['id', 'name', 'type', 'region', 'code', 'bbox', 'geometry']]

locations_global

Skipping field bbox: unsupported OGR type: 3


,id,name,type,region,code,bbox,geometry
0,europe_global,Europe,global,europe,europe,"[-24.84661917981725, 34.66758580473031, 31.585...","MULTIPOLYGON (((-2.41079 35.15382, -2.41093 35..."


In [53]:
locations_admin = gpd.read_file(VECTOR_FOLDER + '/europe_country_boundaries.geojson')
locations_admin.drop(columns=['fid'], inplace=True)
locations_admin['region'] = 'europe'
locations_admin['geometry'] = locations_admin['geometry']. simplify(tolerance=0.005, preserve_topology=True)
locations_admin['bbox'] = locations_admin['geometry'].apply(lambda x: [x.bounds[0], x.bounds[1], x.bounds[2], x.bounds[3]])
locations_admin = locations_admin[['id', 'name', 'type', 'region', 'code', 'bbox', 'geometry']]
locations_admin.head()

Skipping field bbox: unsupported OGR type: 3


,id,name,type,region,code,bbox,geometry
0,europe_adminregion_alb,Albania,admin_region,europe,ALB,"[19.090053751108968, 39.640394043550465, 21.05...","POLYGON ((19.53279 40.03011, 19.354 40.11617, ..."
1,europe_adminregion_and,Andorra,admin_region,europe,AND,"[1.413788159643749, 42.42901958805319, 1.78609...","POLYGON ((1.71173 42.49284, 1.68034 42.49507, ..."
2,europe_adminregion_aut,Austria,admin_region,europe,AUT,"[9.530749050131423, 46.372453491783176, 17.160...","MULTIPOLYGON (((10.45102 47.5546, 10.43086 47...."
3,europe_adminregion_bel,Belgium,admin_region,europe,BEL,"[2.443948864169559, 49.497237871429995, 6.4070...","MULTIPOLYGON (((2.44395 51.2079, 2.63414 51.27..."
4,europe_adminregion_bgr,Bulgaria,admin_region,europe,BGR,"[22.35749624906753, 41.23612650372761, 28.7894...","POLYGON ((28.75633 43.66886, 28.76156 43.61839..."


## Hydrobasins

In [68]:
locations_basin = gpd.read_file(DATA_FOLDER + '/River_basins_ha.gpkg')
locations_basin = locations_basin[['Name & country', 'geometry']]
locations_basin.rename(columns={'Name & country': 'name'}, inplace=True)
locations_basin = locations_basin[~locations_basin['geometry'].is_empty]

#split name and country
locations_basin[['name', 'country']] = locations_basin['name'].str.split(', ', expand=True)
locations_basin['name'] = locations_basin['name'].str.title()
locations_basin['country'] = locations_basin['country'].apply(lambda x: f'({x})' if x else '')
locations_basin['name'] = locations_basin['name'] + ' ' + locations_basin['country']
locations_basin.drop(columns=['country'], inplace=True)
locations_basin['name'] = locations_basin.groupby('name').cumcount().astype(str).radd(locations_basin['name'] + ' ')
locations_basin['name'] = locations_basin['name'].str.replace(' 0', '', regex=False)
locations_basin['name'] = locations_basin['name'].str.replace('2', '3').str.replace('1', '2')
locations_basin['name'] = locations_basin['name'].str.strip()

locations_basin['geometry'] = locations_basin.force_2d()
locations_basin = locations_basin.to_crs(epsg=4326)

print(f'Total entries: {len(locations_basin)}')
print(f'Total unique names: {locations_basin["name"].nunique()}')
display(locations_basin.head())



Total entries: 203
Total unique names: 203


,name,geometry
0,Vistula River Basin Cross Border,"MULTIPOLYGON (((22.54178 49.41633, 22.54178 49..."
1,Tornionjoki River Basin Cross Border,"MULTIPOLYGON (((23.5517 67.24833, 23.5517 67.2..."
2,Tagus River Basin Cross Border,"MULTIPOLYGON (((-6.35062 39.1992, -6.35062 39...."
3,Skagerrak And Kattegat River Basin Cross Border,"MULTIPOLYGON (((11.56201 58.89511, 11.56201 58..."
4,Seine River Basin Cross Border,"MULTIPOLYGON (((-1.84741 48.7334, -1.84737 48...."


In [69]:
# add number to repeated names
locations_basin[locations_basin['name'].str.contains('Scheldt')]['name'].values


array(['Scheldt River Basin Cross Border',
       'International River Basin District Of The Scheldt (BE)',
       'International River Basin District Of The Scheldt (BE) 2',
       'International River Basin District Of The Scheldt (BE) 3',
       'Scheldt (NL)'], dtype=object)

In [70]:
locations_basin['id'] = 'europe_hydrobasin_' + locations_basin['name'].str.replace(' ', '_').str.lower()
locations_basin['type'] = 'hydro_basin'
locations_basin['code'] = 'europe_basin_' + locations_basin.index.astype(str)
locations_basin['region'] = 'europe'
locations_basin['bbox'] = locations_basin['geometry'].apply(lambda x: [x.bounds[0], x.bounds[1], x.bounds[2], x.bounds[3]])

locations_basin = locations_basin[['id', 'name', 'type', 'code', 'region', 'bbox', 'geometry']]
locations_basin.head()

,id,name,type,code,region,bbox,geometry
0,europe_hydrobasin_vistula_river_basin_cross_bo...,Vistula River Basin Cross Border,hydro_basin,europe_basin_0,europe,"[16.838648141575163, 48.999659530683175, 24.14...","MULTIPOLYGON (((22.54178 49.41633, 22.54178 49..."
1,europe_hydrobasin_tornionjoki_river_basin_cros...,Tornionjoki River Basin Cross Border,hydro_basin,europe_basin_1,europe,"[20.548853312751387, 65.61277715958875, 25.259...","MULTIPOLYGON (((23.5517 67.24833, 23.5517 67.2..."
2,europe_hydrobasin_tagus_river_basin_cross_border,Tagus River Basin Cross Border,hydro_basin,europe_basin_2,europe,"[-9.579704756720755, 38.40509736416065, -1.467...","MULTIPOLYGON (((-6.35062 39.1992, -6.35062 39...."
3,europe_hydrobasin_skagerrak_and_kattegat_river...,Skagerrak And Kattegat River Basin Cross Border,hydro_basin,europe_basin_3,europe,"[10.892474600840083, 55.80413254285017, 14.875...","MULTIPOLYGON (((11.56201 58.89511, 11.56201 58..."
4,europe_hydrobasin_seine_river_basin_cross_border,Seine River Basin Cross Border,hydro_basin,europe_basin_4,europe,"[-1.9968307508091663, 46.91909125448744, 5.623...","MULTIPOLYGON (((-1.84741 48.7334, -1.84737 48...."


In [65]:
locations_basin.to_file(VECTOR_FOLDER + '/europe_river_basins.geojson', driver='GeoJSON')

In [42]:
!tippecanoe -o {OUTPUT_MBTILES + '/europe_river_basins.mbtiles'} -zg --extend-zooms-if-still-dropping --force -l europe_river_basins -f {VECTOR_FOLDER + '/europe_river_basins.geojson'}

../data/raw/vectors/europe_river_basins.geojson:194: JSON does not allow NaN: in JSON object {"type":"FeatureCollection","name":"europe_river_basins","crs":{"type":"name","properties":{"name":"urn:ogc:def:crs:OGC:1.3:CRS84"}},"features":[{"type":"Feature","properties":{"id":"europe_hydrobasin_drini_river_basin_(al)","name":"Drini River Basin (AL)","type":"hydro_basin","code":"europe_basin_188","region":"europe","bbox":[]}}]}
188 features, 56365340 bytes of geometry and attributes, 31271 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z0 for features typically 868940 feet (264853 meters) apart, and at least 148947 feet (45399 meters) apart
Choosing a maxzoom of -z13 for resolution of about 57 feet (17 meters) within features
  99.9%  13/4680/1874  


## Combine locations

In [72]:
all_locations = pd.concat([locations_global, locations_admin, locations_basin, locations_pa], ignore_index=True)
all_locations['type'].value_counts()

type
wdpa            44543
hydro_basin       203
admin_region       39
global              1
Name: count, dtype: int64

### Append to the seed locations

In [73]:
LOCATIONS_JSON = '../../app-initial-data/locations.json'
REGION = 'europe'

# Same keys, in the same order, as the records of app-initial-data/locations.json
LOCATION_KEYS = ['id', 'name', 'region', 'code', 'geometry', 'bbox', 'type', 'parent']
# Keeps locations.json small: a bit more simplification than the map tiles and rounded coordinates
LOCATIONS_SIMPLIFY_TOLERANCE = 0.0075
COORDINATE_DECIMALS = 4

locations_export = gpd.GeoDataFrame(all_locations, geometry='geometry', crs='EPSG:4326')
locations_export['code'] = locations_export['code'].astype(str)
locations_export['parent'] = None
locations_export['geometry'] = locations_export['geometry'].simplify(LOCATIONS_SIMPLIFY_TOLERANCE, preserve_topology=True)
locations_export['geometry'] = shapely.transform(
    locations_export['geometry'].values, lambda coords: np.round(coords, COORDINATE_DECIMALS)
)
locations_export['bbox'] = locations_export['geometry'].apply(lambda x: list(x.bounds))

assert locations_export['id'].is_unique, 'Duplicate location IDs'
assert locations_export['name'].notna().all(), 'Locations without a name'
assert not locations_export['geometry'].is_empty.any(), 'Locations with an empty geometry'
assert locations_export['id'].str.startswith(f'{REGION}_').all(), f'Location IDs must start with {REGION}_'

features = json.loads(locations_export.to_json(na='null', drop_id=True))['features']
new_locations = [
    {key: {**f['properties'], 'geometry': f['geometry']}[key] for key in LOCATION_KEYS}
    for f in features
]

print(f'New locations: {len(new_locations)}')
locations_export['type'].value_counts()

New locations: 44786


type
wdpa            44543
hydro_basin       203
admin_region       39
global              1
Name: count, dtype: int64

### Country names translations

In [ ]:
# English and French country names, so that re-running the notebook keeps the translations
COUNTRY_NAMES_CSV = '../data/country-names-translations.csv'

country_names = pd.read_csv(COUNTRY_NAMES_CSV)
country_names = country_names[country_names['region'] == REGION].set_index('id')

for location in new_locations:
    if location['id'] in country_names.index:
        names = country_names.loc[location['id']]
        location['name'] = names['name_en']
        location['locale'] = {'fr': {'name': names['name_fr']}}

countries = [l['id'] for l in new_locations if l['type'] == 'admin_region']
missing = [c for c in countries if c not in country_names.index]
assert not missing, f'Countries without translations in {COUNTRY_NAMES_CSV}: {missing}'
print(f'Countries with translated names: {len(countries)}')

In [74]:
with open(LOCATIONS_JSON, encoding='utf-8') as f:
    locations = json.load(f)

# Drops the region's locations from a previous run, so that the cell can be run again
locations = [l for l in locations if l['region'] != REGION]
existing_ids = {l['id'] for l in locations}
clashing_ids = [l['id'] for l in new_locations if l['id'] in existing_ids]
assert not clashing_ids, f'IDs already in {LOCATIONS_JSON}: {clashing_ids[:5]}'

locations += new_locations

# Same formatting as the app's export script (JSON.stringify(data, null, 2)), so the existing records don't change
with open(LOCATIONS_JSON, 'w', encoding='utf-8') as f:
    json.dump(locations, f, indent=2, ensure_ascii=False)

pd.Series([l['region'] for l in locations]).value_counts()

europe    44786
sahel      2068
Name: count, dtype: int64